# MCP (Model Context Protocol) Crash Course Notes

Personal revision notes based on the MCP-CRASH-Course repo by Krish Naik, with explanations, analogies, and runnable code examples.


## What is MCP?

**MCP (Model Context Protocol)** is a standard way to let an AI model (like Claude or GPT) talk to outside tools, APIs, and data — without writing custom glue code for every single AI app.

### Analogy: MCP is like a USB port for AI

* Before USB, every device (mouse, printer, camera) needed its own special cable and driver.
* USB gave everyone one universal port — plug in any device, it just works.
* MCP does this for AI: build **one MCP server** for weather, and *any* MCP-compatible AI app (Claude Desktop, VS Code Copilot Chat, a custom agent) can plug into it instantly.

There are always two sides:

* **MCP Server** — exposes capabilities (tools, resources) to the outside world.
* **MCP Client** — the AI app that connects to the server and uses those capabilities.


## Creating an MCP Server

`FastMCP` is a helper class that handles all the MCP protocol boilerplate (handshakes, message formats, etc.) so you just focus on writing functions.

**Real-world example:** Registering a new plugin — you just say "here's my server, name it `weather`", instead of writing raw protocol-handling code.


In [ ]:
# Install once: pip install "mcp[cli]" httpx

from typing import Any
import httpx
from mcp.server.fastmcp import FastMCP

# Create an MCP server named 'weather'
mcp = FastMCP("weather")

NWS_API_BASE = "https://api.weather.gov"
USER_AGENT = "weather-app/1.0"


## Tools — the "actions" an AI can take

A **Tool** is a function the AI model is *allowed to call* when it decides it needs to. The docstring is not just documentation — the AI actually reads it to decide **when** and **how** to use the tool.

**Real-world example:** A customer asks a chatbot, "Should I cancel my flight to Florida?" The model reasons it needs current weather alerts for Florida, sees the `get_alerts(state)` tool description, calls `get_alerts("FL")`, and answers using real data instead of guessing.


In [ ]:
async def make_nws_request(url: str) -> dict[str, Any] | None:
    """Make a request to the NWS API with proper error handling."""
    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "application/geo+json"
    }
    async with httpx.AsyncClient() as client:
        try:
            response = await client.get(url, headers=headers, timeout=30.0)
            response.raise_for_status()
            return response.json()
        except Exception:
            return None


def format_alert(feature: dict) -> str:
    """Format an alert feature into a readable string."""
    props = feature["properties"]
    return f"""
    Event: {props.get('event', 'Unknown')}
    Area: {props.get('areaDesc', 'Unknown')}
    Severity: {props.get('severity', 'Unknown')}
    Description: {props.get('description', 'No description available')}
    Instructions: {props.get('instruction', 'No specific instructions provided')}
    """


@mcp.tool()
async def get_alerts(state: str) -> str:
    """Get weather alerts for a US state.

    Args:
        state: Two-letter US state code (e.g. CA, NY)
    """
    url = f"{NWS_API_BASE}/alerts/active/area/{state}"
    data = await make_nws_request(url)

    if not data or "features" not in data:
        return "Unable to fetch alerts or no alerts found."

    if not data["features"]:
        return "No active alerts for this state."

    alerts = [format_alert(feature) for feature in data["features"]]
    return "\n---\n".join(alerts)


@mcp.tool()
async def get_forecast(latitude: float, longitude: float) -> str:
    """Get weather forecast for a location.

    Args:
        latitude: Latitude of the location
        longitude: Longitude of the location
    """
    points_url = f"{NWS_API_BASE}/points/{latitude},{longitude}"
    points_data = await make_nws_request(points_url)

    if not points_data:
        return "Unable to fetch forecast data for this location."

    forecast_url = points_data["properties"]["forecast"]
    forecast_data = await make_nws_request(forecast_url)

    if not forecast_data:
        return "Unable to fetch detailed forecast."

    periods = forecast_data["properties"]["periods"]
    forecasts = []
    for period in periods[:5]:  # Only show next 5 periods
        forecast = f"""
                {period['name']}:
                Temperature: {period['temperature']}°{period['temperatureUnit']}
                Wind: {period['windSpeed']} {period['windDirection']}
                Forecast: {period['detailedForecast']}
                """
        forecasts.append(forecast)

    return "\n---\n".join(forecasts)


### Bonus — Simple Tools (from the basic example)

Before jumping into a real API, it helps to see the *shape* of a tool with something trivial.


In [ ]:
@mcp.tool()
def add(a: int, b: int) -> int:
    """Adds two numbers"""
    return a + b


@mcp.tool()
def subtract(a: int, b: int) -> int:
    """Subtracts second number from first"""
    return a - b


## Resources — the "read-only data" an AI can fetch

A **Resource** is a URI-addressable piece of data — like a webpage the AI can "visit" to read something, rather than an action it performs. `{message}` and `{name}` work like route parameters in a web API (e.g. `/users/{id}`).

**Real-world example:** In a company AI assistant, you might expose `employee://{id}` as a resource so the AI can "look up" employee record #452 the same way a browser loads a URL — read-only, no side effects. Compare this to a **Tool** like `send_email()`, which *does something* rather than just fetching info.


In [ ]:
@mcp.resource("echo://{message}")
def echo_resource(message: str) -> str:
    """Echo a message as a resource"""
    return f"Resource echo: {message}"


@mcp.resource("greeting://{name}")
def greeting_resource(name: str) -> str:
    """Returns a personalized greeting"""
    return f"Hello, {name}! Welcome to the MCP server."


## Running the Server

MCP servers can run with different **transports** — the pipes that carry messages between client and server.


In [ ]:
# Run the server
if __name__ == "__main__":
    transport = "stdio"  # or "sse"
    if transport == "stdio":
        print("Running server with stdio transport")
        mcp.run(transport="stdio")
    elif transport == "sse":
        print("Running server with SSE transport")
        mcp.run(transport="sse")
    else:
        raise ValueError(f"Unknown transport: {transport}")


## Transport 1: stdio (local subprocess pipe)

The client **launches the server as a subprocess** and talks to it via standard input/output — no network ports involved.

**Real-world example:** This is like Claude Desktop launching a local MCP server on your own laptop when you open the app — it starts the process quietly in the background and pipes messages back and forth. Great for local tools (e.g. "read my local files").

**Rule of thumb:** stdio = local, single-user, simple.


In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()  # Needed to run interactive python (e.g. notebooks)


async def main():
    # Define server parameters — how to launch the server
    server_params = StdioServerParameters(
        command="python",       # The command to run your server
        args=["server.py"],     # Arguments to the command
    )

    # Connect to the server
    async with stdio_client(server_params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            # 1. Handshake with the server
            await session.initialize()

            # 2. Ask: 'what can you do?'
            tools_result = await session.list_tools()
            print("Available tools:")
            for tool in tools_result.tools:
                print(f"  - {tool.name}: {tool.description}")

            # 3. Actually call a tool
            result = await session.call_tool("get_alerts", arguments={"state": "CA"})
            print(f"The weather alerts are = {result.content[0].text}")


if __name__ == "__main__":
    asyncio.run(main())


## Transport 2: SSE (networked HTTP server)

SSE (Server-Sent Events) means the server runs as a standalone **web service** on a port, and the client connects to it over HTTP, like connecting to any web API.

**Real-world example:** A company hosts one central "Weather MCP Server" on a cloud VM. Fifty different internal AI apps (a Slack bot, a support dashboard, a chatbot) can all connect to that *same* running server over the network, instead of each needing its own local copy.

**Rule of thumb:** SSE = networked, shareable, production-ready.


In [ ]:
# Server side (mcpserver/server.py) — expose host/port for SSE
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(
    name="weather",
    host="0.0.0.0",  # only used for SSE transport (localhost)
    port=8000,       # only used for SSE transport (set this to any port)
)


In [ ]:
# Client side (client-sse.py)
import asyncio
import nest_asyncio
from mcp import ClientSession
from mcp.client.sse import sse_client

nest_asyncio.apply()  # Needed to run interactive python

"""
Make sure:
1. The server is running before running this script.
2. The server is configured to use SSE transport.
3. The server is listening on port 8000.
"""


async def main():
    # Connect to the server using SSE
    async with sse_client("http://localhost:8000/sse") as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            print("Available tools:")
            for tool in tools_result.tools:
                print(f"  - {tool.name}: {tool.description}")

            result = await session.call_tool("get_alerts", arguments={"state": "CA"})
            print(f"The weather alerts are = {result.content[0].text}")


if __name__ == "__main__":
    asyncio.run(main())


## Containerizing the Server (Dockerfile)

Once a server uses SSE (networked), it makes sense to package it in Docker so it can be deployed anywhere reliably.


In [ ]:
# Dockerfile (not Python — shown here for revision)
dockerfile_contents = '''
FROM python:3.11-slim

WORKDIR /demo

# Copy requirements file
COPY requirements.txt .

# Install dependencies using uv
RUN pip install uv
RUN uv venv
RUN uv pip install -r requirements.txt

# Copy application code
COPY server.py .
COPY client-sse.py .

# Expose the port the server runs on
EXPOSE 8000

# Command to run the server
CMD ["uv", "run", "server.py"]
'''
print(dockerfile_contents)


## Wiring an Actual LLM Agent to the MCP Server

This ties everything together into a real autonomous agent:

1. A config file tells the client **how to launch the MCP server**.
2. An actual LLM is plugged in as "the brain".
3. `MCPAgent` combines the LLM + MCP tools into an agent loop: the LLM reads the question, decides if it needs a tool, calls it via MCP, reads the result, and keeps reasoning (`max_steps` caps the loop) — with `memory_enabled=True` so it remembers earlier turns.

**Real-world example:** A dispatcher at a logistics company types: *"Any alerts in Texas today, and should we delay the Austin truck route?"* The LLM reasons it needs live data, calls the MCP `get_alerts("TX")` tool, gets real severe weather warnings, and combines that with its own reasoning to make an informed recommendation — remembering the conversation if the dispatcher follows up with "what about tomorrow?"


In [ ]:
# weather.json — MCP server launch config for the client
weather_json = {
    "mcpServers": {
        "weather": {
            "command": "uv",
            "args": [
                "run",
                "--with",
                "mcp[cli]",
                "mcp",
                "run",
                "server/weather.py"
            ]
        }
    }
}
print(weather_json)


In [ ]:
import asyncio
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from mcp_use import MCPAgent, MCPClient


async def run_memory_chat():
    """Run a chat using MCPAgent's built-in conversation memory."""
    # Load environment variables for API keys
    load_dotenv()
    os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

    # Config file path - change this to your config file
    config_file = "server/weather.json"

    print("Initializing chat...")

    # Create MCP client and agent with memory enabled
    client = MCPClient.from_config_file(config_file)
    llm = ChatGroq(model="qwen-qwq-32b")

    # Create agent with memory_enabled=True
    agent = MCPAgent(
        llm=llm,
        client=client,
        max_steps=15,
        memory_enabled=True,  # Enable built-in conversation memory
    )

    print("\n===== Interactive MCP Chat =====")
    print("Type 'exit' or 'quit' to end the conversation")
    print("Type 'clear' to clear conversation history")
    print("==================================\n")

    try:
        while True:
            user_input = input("\nYou: ")

            if user_input.lower() in ["exit", "quit"]:
                print("Ending conversation...")
                break

            if user_input.lower() == "clear":
                agent.clear_conversation_history()
                print("Conversation history cleared.")
                continue

            print("\nAssistant: ", end="", flush=True)
            try:
                response = await agent.run(user_input)
                print(response)
            except Exception as e:
                print(f"\nError: {e}")
    finally:
        if client and client.sessions:
            await client.close_all_sessions()


if __name__ == "__main__":
    asyncio.run(run_memory_chat())


## Quick Recap Table

| Concept | What it is | File(s) | Real-world parallel |
|---|---|---|---|
| MCP Server | Exposes tools/resources | `weather.py`, `server.py` | A USB device |
| Tool | Callable action | `get_alerts`, `get_forecast`, `add`/`subtract` | "Do something" button |
| Resource | Readable data via URI | `greeting://{name}`, `echo://{message}` | A webpage you visit |
| stdio transport | Local subprocess pipe | `client-stdio.py` | App launching a helper program locally |
| SSE transport | Networked HTTP server | `client-sse.py`, `mcpserver/server.py` | A hosted API anyone can call |
| Dockerfile | Containerize the server | `mcpserver/Dockerfile` | Shipping the server anywhere reliably |
| MCPAgent + LLM | Full reasoning agent | `server/client.py` | The actual chatbot users talk to |

### End-to-end flow

`raw API` → `MCP tool` → `MCP server` → `transport (stdio/SSE)` → `client` → `LLM agent` → `real business decision`
